In [1]:
import numpy as np
import random
import math

# --- 1. PROBLEM DEFINITION: Smart Grid Environment ---

# Define appliances with their power consumption (in kW) and required runtime (in hours)
# and the window of time (hour of day, 0-23) they are allowed to run.
APPLIANCES = {
    'Washing Machine': {'power': 2.0, 'runtime': 2, 'window': (0, 6)},      # Can run between midnight and 6 AM
    'Dishwasher':      {'power': 1.5, 'runtime': 3, 'window': (20, 23)},   # Can run between 8 PM and 11 PM
    'EV Charger':      {'power': 7.0, 'runtime': 4, 'window': (22, 6)},    # Can run between 10 PM and 6 AM (overnight)
    'Water Heater':    {'power': 4.0, 'runtime': 1, 'window': (3, 7)},      # Can run between 3 AM and 7 AM
    'Pool Pump':       {'power': 1.0, 'runtime': 5, 'window': (0, 23)},     # Can run anytime
}

# Time-of-Use (TOU) electricity pricing (cost per kWh)
# Prices are higher during peak demand hours.
PRICE_TARIFF = {
    (0, 6):   0.12,  # Off-Peak: 12 AM - 6 AM
    (7, 17):  0.20,  # Mid-Peak: 7 AM - 5 PM
    (18, 21): 0.35,  # On-Peak:  6 PM - 9 PM
    (22, 23): 0.20,  # Mid-Peak: 10 PM - 11 PM
}

def get_price(hour):
    """Returns the electricity price for a given hour."""
    for (start, end), price in PRICE_TARIFF.items():
        if start <= hour <= end:
            return price
    return float('inf') # Should not happen with a complete tariff

def calculate_cost(schedule):
    """
    Fitness function: Calculates the total electricity cost for a given schedule.
    A 'schedule' is a list of integer start times for each appliance.
    """
    total_cost = 0
    appliance_names = list(APPLIANCES.keys())
    
    for i, start_time in enumerate(schedule):
        appliance = APPLIANCES[appliance_names[i]]
        power = appliance['power']
        runtime = appliance['runtime']
        
        # Calculate cost for the duration the appliance runs
        for hour_offset in range(runtime):
            # Use modulo 24 to handle schedules that wrap around midnight
            current_hour = (start_time + hour_offset) % 24
            total_cost += power * get_price(current_hour)
            
    return total_cost

def generate_random_schedule():
    """Generates a single valid random schedule (a nest)."""
    schedule = []
    for name, details in APPLIANCES.items():
        start_window, end_window = details['window']
        runtime = details['runtime']
        
        if start_window <= end_window:
            # Simple window (e.g., 8 AM to 5 PM)
            possible_starts = list(range(start_window, end_window - runtime + 2))
        else:
            # Window wraps around midnight (e.g., 10 PM to 6 AM)
            possible_starts = list(range(start_window, 24 - runtime + 1)) + list(range(0, end_window - runtime + 2))

        if not possible_starts:
             # Fallback if runtime is too long for the window
             possible_starts = [start_window]

        start_time = random.choice(possible_starts)
        schedule.append(start_time)
        
    return schedule

# --- 2. CUCKOO SEARCH ALGORITHM IMPLEMENTATION ---

def levy_flight(beta=1.5):
    """
    Generate a step length from a Levy distribution.
    This simulates the random walk pattern of cuckoos.
    """
    sigma = (math.gamma(1 + beta) * math.sin(math.pi * beta / 2) / 
             (math.gamma((1 + beta) / 2) * beta * 2**((beta - 1) / 2)))**(1 / beta)
    u = np.random.normal(0, sigma)
    v = np.random.normal(0, 1)
    step = u / abs(v)**(1 / beta)
    return step

def apply_bounds(new_start_time, appliance_name):
    """Ensure the new start time is valid for the appliance's window."""
    details = APPLIANCES[appliance_name]
    start_window, end_window = details['window']
    runtime = details['runtime']

    # Simple validation: wrap around using modulo and clamp if necessary
    # A more complex system might find the nearest valid time slot.
    new_start_time = int(round(new_start_time)) % 24

    if start_window <= end_window:
        if not (start_window <= new_start_time <= end_window - runtime + 1):
            return start_window # Reset to a known valid start
    else: # Wraps around midnight
        if not ((start_window <= new_start_time <= 23) or (0 <= new_start_time <= end_window - runtime + 1)):
            return start_window # Reset to a known valid start

    return new_start_time


def cuckoo_search_optimizer(n_nests, n_iterations, pa):
    """
    Main Cuckoo Search optimization function.
    
    :param n_nests: Number of nests in the population.
    :param n_iterations: Number of generations to run.
    :param pa: Fraction of nests to be abandoned and replaced.
    """
    # Initialize population of nests
    nests = [generate_random_schedule() for _ in range(n_nests)]
    costs = [calculate_cost(s) for s in nests]
    
    best_nest_index = np.argmin(costs)
    best_cost = costs[best_nest_index]
    best_schedule = nests[best_nest_index]
    
    print(f"Initial best schedule cost: ${best_cost:.2f}")

    # Main optimization loop
    for t in range(n_iterations):
        # 1. Get a cuckoo (generate a new solution via Levy flights)
        # Randomly choose a nest to modify
        i = random.randint(0, n_nests - 1)
        current_nest = nests[i][:]
        
        # Generate new solution
        step_size = 0.1 # A scaling factor for the levy flight step
        new_nest = current_nest[:]
        
        # Modify one appliance's start time in the chosen nest
        app_to_modify = random.randint(0, len(APPLIANCES) - 1)
        app_name = list(APPLIANCES.keys())[app_to_modify]
        
        new_start = new_nest[app_to_modify] + step_size * levy_flight()
        new_nest[app_to_modify] = apply_bounds(new_start, app_name)
        
        new_cost = calculate_cost(new_nest)
        
        # 2. Compare the new solution with a randomly chosen existing one
        j = random.randint(0, n_nests - 1)
        if new_cost < costs[j]:
            nests[j] = new_nest
            costs[j] = new_cost
            
        # 3. Abandon a fraction (pa) of the worst nests and build new ones
        n_abandon = int(pa * n_nests)
        worst_indices = np.argsort(costs)[-n_abandon:]
        
        for idx in worst_indices:
            nests[idx] = generate_random_schedule()
            costs[idx] = calculate_cost(nests[idx])
            
        # 4. Find the current best solution
        current_best_idx = np.argmin(costs)
        if costs[current_best_idx] < best_cost:
            best_cost = costs[current_best_idx]
            best_schedule = nests[current_best_idx]
            
        if (t + 1) % 10 == 0:
            print(f"Iteration {t+1}/{n_iterations} | Best Cost: ${best_cost:.2f}")

    return best_schedule, best_cost

# --- 3. RUN SIMULATION ---
if __name__ == "__main__":
    # Cuckoo Search Parameters
    NUM_NESTS = 50
    ITERATIONS = 100
    ABANDON_FRACTION = 0.25

    print("--- Cuckoo Search for Smart Grid Load Scheduling ---")
    
    # Calculate cost of a non-optimized, naive schedule (running everything as early as possible)
    naive_schedule = []
    for name, details in APPLIANCES.items():
        naive_schedule.append(details['window'][0])
    naive_cost = calculate_cost(naive_schedule)
    print(f"\nCost of a NAIVE schedule (running at earliest possible time): ${naive_cost:.2f}\n")
    
    print("Running Cuckoo Search Optimizer...")
    
    optimized_schedule, optimized_cost = cuckoo_search_optimizer(
        n_nests=NUM_NESTS,
        n_iterations=ITERATIONS,
        pa=ABANDON_FRACTION
    )
    
    print("\n--- Optimization Complete ---")
    print(f"Optimized Schedule Cost: ${optimized_cost:.2f}")
    
    print("\nOptimal Appliance Schedule:")
    appliance_names = list(APPLIANCES.keys())
    for i, start_time in enumerate(optimized_schedule):
        appliance_name = appliance_names[i]
        runtime = APPLIANCES[appliance_name]['runtime']
        end_time = (start_time + runtime -1) % 24
        print(f"- {appliance_name:<16}: Start at {start_time:02d}:00, End at {end_time:02d}:00")

    print(f"\nTotal savings compared to naive schedule: ${naive_cost - optimized_cost:.2f}")


--- Cuckoo Search for Smart Grid Load Scheduling ---

Cost of a NAIVE schedule (running at earliest possible time): $7.39

Running Cuckoo Search Optimizer...
Initial best schedule cost: $6.05
Iteration 10/100 | Best Cost: $6.05
Iteration 20/100 | Best Cost: $6.05
Iteration 30/100 | Best Cost: $6.05
Iteration 40/100 | Best Cost: $6.05
Iteration 50/100 | Best Cost: $6.05
Iteration 60/100 | Best Cost: $6.05
Iteration 70/100 | Best Cost: $6.05
Iteration 80/100 | Best Cost: $6.05
Iteration 90/100 | Best Cost: $6.05
Iteration 100/100 | Best Cost: $6.05

--- Optimization Complete ---
Optimized Schedule Cost: $6.05

Optimal Appliance Schedule:
- Washing Machine : Start at 01:00, End at 02:00
- Dishwasher      : Start at 21:00, End at 23:00
- EV Charger      : Start at 01:00, End at 04:00
- Water Heater    : Start at 06:00, End at 06:00
- Pool Pump       : Start at 00:00, End at 04:00

Total savings compared to naive schedule: $1.34
